# Exercise 1 - Exploratory data analysis (EDA) - portable notebook

This is the **portable version** of the Chapter 1 EDA practice from
**Machine Learning for Neuroscience**, generated from the canonical
course notebook by `scripts/build_portable_notebook.py`. It is meant for
running or editing the code in Google Colab or in a local VS Code /
Jupyter setup.

The richer version -- with the four activities embedded and running in
the browser -- is the published course page:
<https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_01/exercise_01.html>

In this notebook the four interactive activities are replaced by links
to that page; every Python analysis cell is kept and runnable. Questions
marked *Think first* are followed, where one exists, by a collapsible
*Check your reasoning* block; open questions are left without one fixed
answer.

## Setup

This notebook imports only `numpy`, `pandas`, `matplotlib` and
`seaborn`. All four are already installed on Google Colab, and in a
typical scientific-Python environment, so there is normally nothing to
do here.

If one of the imports further down fails, run the next cell once (edit
the version pins if your project needs specific ones), then restart the
kernel and run the notebook from the top. The notebook also downloads a
public data file the first time it runs, so it needs internet access.

In [ ]:
# If an import below fails, uncomment and run this line once, then
# restart the kernel. Safe on Colab, VS Code and Jupyter.
# %pip install numpy pandas matplotlib seaborn

# Exercise 1 — Exploratory data analysis (EDA)

This exercise applies **exploratory data analysis (EDA)** to the phenotypic
table of the Autism Brain Imaging Data Exchange II (ABIDE-II). We inspect the
table's structure, convert coded variables to categories, quantify and reason
about missing data, describe and compare distributions, and read correlations
together with the sample size behind each number. The aim is not a predictive
model but a defensible understanding of the data before any modelling begins.

## What this notebook covers

1. **Importing and data loading** — a small curated slice of the ABIDE-II
   phenotypic table.
2. **Data inspection** — comparing `head()`, `tail()`, and `sample()`, and
   spotting categories that are stored as numbers.
3. **Statistical inspection** — `info()` and `describe()`, and their blind
   spots.
4. **Missing values** — quantifying missingness, its pattern across sites and
   diagnosis, and strategies for handling it (with the complete-case
   retention explorer).
5. **Distributions and feature correlations** — reading one distribution,
   comparing across groups, range checks, and Pearson/Spearman correlations
   (with the correlation explorer).

**Prerequisites:** basic Python and a first acquaintance with `pandas`,
`matplotlib`, and `seaborn`. Run the cells in order from the top; the data is
read from a pinned public URL, so the first run needs internet access.

## 1. Importing and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

<details>
<summary><strong>Loading the ABIDE-II phenotypic data</strong></summary>

The [Autism Brain Imaging Data Exchange II (ABIDE-II)](https://fcon_1000.projects.nitrc.org/indi/abide/abide_II.html) combines neuroimaging and phenotypic data collected across 19 international research sites. The full phenotypic table has 348 variables — demographics, diagnostic assessments, cognitive scores, behavioral questionnaires, and acquisition-related fields.

For this exercise we use a deliberately small curated slice: 13 columns covering identifiers, core demographics, diagnosis, a few cognitive scores, and a couple of autism-assessment totals. The meaning and coding of every variable are documented in the official [ABIDE-II Phenotypic Data Legend](https://fcon_1000.projects.nitrc.org/indi/abide/ABIDEII_Data_Legend.pdf).

Some variables are recorded for nearly every participant; others were collected only at particular sites or for particular participant groups.

</details>

In [ ]:
PHENOTYPES_URL = (
    "https://raw.githubusercontent.com/"
    "neurohackademy/nh2020-curriculum/"
    "e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b/"
    "tu-machine-learning-yarkoni/data/abide2_phenotypic.csv"
)

# Curated column subset, embedded so this notebook needs no repository
# files (the canonical Jupyter Book notebook reads it from book/config/).
CURATED_COLUMNS = [
    "SITE_ID",
    "SUB_ID",
    "DX_GROUP",
    "AGE_AT_SCAN",
    "SEX",
    "HANDEDNESS_CATEGORY",
    "FIQ",
    "VIQ",
    "PIQ",
    "CURRENT_MED_STATUS",
    "SRS_TOTAL_RAW",
    "ADOS_G_TOTAL",
    "ADI_R_SOCIAL_TOTAL_A",
]

phenotypes = pd.read_csv(PHENOTYPES_URL, encoding="latin-1", low_memory=False)
phenotypes.columns = phenotypes.columns.str.strip()
phenotypes = phenotypes[CURATED_COLUMNS].copy()

rows, cols = phenotypes.shape
print(f"Data table shape: {rows,cols}")

## 2. Data inspection

Before computing any statistic, it is worth looking at a few individual
rows. This shows how the table is organised, how each variable is
represented, and whether anything looks unexpected.

pandas offers several ways to pull out rows to look at:

- `phenotypes.head(n)` — the first `n` rows;
- `phenotypes.tail(n)` — the last `n` rows;
- `phenotypes.sample(n)` — `n` rows chosen at random;
- a custom selection — particular rows, evenly spaced rows, or rows that
  meet a condition.

Which of these gives the most useful first impression of a table? The
activity below lets you compare them on this dataset before we draw any
conclusion.

The interactive comparison lets you switch between `head()`, `tail()`
and `sample()`, change the row count, and see how many acquisition sites
and missing cells each view contains. The next cell runs the same three
views in Python.

> **Interactive version on the course website.** It is embedded in the
> published Chapter 1 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_01/exercise_01.html>
> This portable notebook links to it instead of embedding it.

#### Think first

Compare the three views at the same row count.

1. For `head()` and for `tail()`, how many acquisition sites appear? What does that tell you about how the table is ordered?
2. Switch to `sample()`. How many sites appear now, and how does the number of missing cells compare with `head()` and `tail()`?
3. Which view would you trust more for a first, broad impression of the whole dataset — and what is the one thing a small random sample still cannot promise you?

`head()` and `tail()` are **deterministic**: they always return the same
rows, which makes them the right tool for checking the column layout, the
data types, and how the table is sorted. In this table the rows are grouped
by acquisition site, so `head()` and `tail()` each land inside a single
site.

`sample()` draws rows from across the whole table, so in an ordered,
multi-site table like this one it usually shows a wider spread of sites and
a more representative mix of missing and recorded values. It is still only a
sample, though: a handful of rows is not guaranteed to reflect the full
dataset, and a different random draw would show different participants.
Passing `random_state` just makes one particular draw reproducible.

In [ ]:
# The same three views in Python. head() and tail() are deterministic;
# sample() takes random_state, so a chosen seed reproduces the same draw.
from IPython.display import display

display(phenotypes.head(8))
display(phenotypes.tail(8))
display(phenotypes.sample(8, random_state=0))

#### Think first

Look at the rows and the column names in the views above before continuing.

1. **What does each row represent?**

2. **Which variables are categorical?**  
   Remember that categorical variables are not necessarily stored as text. Some categories may be represented using numerical codes.

3. **Are there variables or participants that already look problematic?**

Answer before inspecting the complete data summary.

## 3. Statistical inspection

Looking at individual rows helps us understand the structure of a dataset, but it does not summarize the dataset as a whole. Pandas provides two useful starting points:

- `DataFrame.info()` summarizes the dataset's structure and data types.
- `DataFrame.describe()` calculates descriptive statistics for each variable.

These methods answer different questions and should usually be used together.

### Dataset structure with `info()`

`info()` reports:

- the number of rows and columns;
- each column's data type;
- the number of non-missing observations;
- an estimate of memory usage.

It is the quick way to check the shape, the data types, and the non-null
counts, and to spot a variable stored as an unexpected type — but it says
nothing about the *distribution* of the values inside a column.

In [ ]:
phenotypes.info()

#### Think first

Examine the output of `phenotypes.info()`.

1. How many participants and variables are present?
2. Which variables contain missing observations?
3. Are any categorical variables stored as numbers/strings?

### Identifying hidden categorical variables

A dataframe's **storage type** is not necessarily the variable's **statistical type**.

For example, pandas initially reads:

- `SITE_ID` as text (`str`);
- `DX_GROUP` and `SEX` as integers;
- `HANDEDNESS_CATEGORY` and `CURRENT_MED_STATUS` as floating-point numbers, only because they contain missing values.

Nevertheless, all of these variables represent categories rather than numerical measurements. Their numerical codes are labels: arithmetic operations such as calculating their mean are generally not meaningful.

Before calculating descriptive statistics, we should explicitly tell pandas which variables are categorical.

#### Think first

**Identify the variables.**

Using the column names and the [ABIDE-II Phenotypic Data Legend](https://fcon_1000.projects.nitrc.org/indi/abide/ABIDEII_Data_Legend.pdf), consider:

1. Which numerical columns represent category codes?
2. Which text columns represent a limited set of categories?
3. Should `SUB_ID` be considered a measurement, a category, or an identifier?
4. `HANDEDNESS_CATEGORY` is coded `1`, `2`, `3`. Is that an ordered scale or three unordered groups? How would you check?

In [ ]:
categorical_columns = [
    "SITE_ID",
    "DX_GROUP",
    "SEX",
    "HANDEDNESS_CATEGORY",
    "CURRENT_MED_STATUS",
]

# Defining the dtype (data type) to be categorical:
phenotypes[categorical_columns] = phenotypes[categorical_columns].astype("category")

# Participant IDs are labels, not numerical measurements.
phenotypes["SUB_ID"] = phenotypes["SUB_ID"].astype("string")

The `category` dtype tells pandas that these columns contain a finite set of possible groups or labels. Missing values remain missing after the conversion.

The conversion does not change the meaning of the original codes. For example, the values `1` and `2` in `DX_GROUP` remain `1` and `2`; pandas now simply knows that they represent categories rather than quantities with a meaningful distance between them.

None of these variables are declared as *ordered* categories. `HANDEDNESS_CATEGORY`, for instance, codes right / left / mixed handedness as `1` / `2` / `3` — three groups, not an increasing scale. When the meaning of a coded variable is unclear, consult the study's data legend or the people who collected it rather than guessing.

In [ ]:
phenotypes[categorical_columns + ["SUB_ID"]].dtypes

#### Why does this matter?

Correctly assigning categorical data types:

- prevents coded categories from appearing in numerical summaries;
- makes categorical summaries more informative;
- helps plotting libraries treat categories as distinct groups;
- makes the intended meaning of each variable explicit.

However, pandas' categorical dtype does not automatically prepare a variable for machine-learning models. Later, categorical predictors may require an encoding method such as one-hot encoding (later in the course).

### Numerical summaries with `describe()`

By default, `describe()` summarizes numerical columns. Its output includes:

- `count`: number of non-missing observations;
- `mean`: arithmetic mean;
- `std`: sample standard deviation;
- `min` and `max`: smallest and largest observed values;
- `25%`, `50%`, and `75%`: quartiles of the distribution.

Transposing the output (using `.T`) places variables in rows, which is often easier to read.

`describe()` summarises the distribution of each numeric variable, but a single row of numbers can hide differences between groups or sites, and the `count` column is the reminder that each statistic is computed only from the observations that are actually present.

In [ ]:
phenotypes.describe().T
# And for the categorical variables you can use: 
# phenotypes.describe(include=["category"]).T

#### Think first

Use the output above to answer the following questions.

1. Which variables have the most missing data?
2. Find one variable for which the mean and median differ noticeably. What might cause this difference?
3. Do any minimum or maximum values appear biologically or clinically surprising?
4. Why is calculating a mean for `DX_GROUP` or `SEX` usually not meaningful?
5. Could two variables have identical means and standard deviations but very different distributions?

## 4. Missing values

A missing value indicates that no usable value is available for a particular participant and variable.
In pandas, missing values are usually represented as `NaN` or `<NA>`.

A measurement may be missing because it was not collected, was not applicable, failed quality control, or was unavailable at a particular acquisition site.

Before deciding how to handle missing values, we should ask:

1. How much data is missing?
2. Which variables and participants are affected?
3. Is the missingness associated with site, diagnosis, or another variable?
4. Which variables are actually required for our analysis?

### Quantifying missing values

In [ ]:
missing_summary = pd.DataFrame(
    {
        "n_missing": phenotypes.isna().sum(),
        "percent_missing": phenotypes.isna().mean() * 100,
    }
).sort_values("percent_missing", ascending=False)

missing_summary.query("n_missing > 0").round(1)

The percentage of missing observations is calculated separately for each variable:

$$
\text{Percentage missing}
= \frac{\text{Number of missing observations}}
{\text{Total number of observations}}
\times 100
$$

A large percentage does not automatically mean that a variable should be removed. A sparsely measured variable could still be scientifically important.

#### Think first

1. Which variables contain no missing values?
2. Which variables have more than 50% missing observations?
3. Would you automatically remove a variable because it has more than 50% missing data? Why or why not?
4. Does this table tell us why the values are missing?

### Is missingness distributed randomly?

ABIDE-II combines data from multiple research sites. Different sites may have administered different questionnaires or used different assessment procedures.

Consequently, missingness may be related to the acquisition site. If so, removing incomplete observations could change the representation of sites in the dataset.

In [ ]:
top_missing_variables = (
    missing_summary
    .query("n_missing > 0")
    .head(15)
    .index
)

missing_by_site = (
    phenotypes[top_missing_variables]
    .isna()
    .groupby(phenotypes["SITE_ID"], observed=True)
    .mean()
    .mul(100)
)

plt.figure(figsize=(14, 7))

sns.heatmap(
    missing_by_site.T,
    cmap="coolwarm",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    linecolor="black",
    cbar_kws={"label": "Missing observations (%)"},
    
)

plt.xlabel("Acquisition site")
plt.ylabel("Variable")
plt.title("Missingness by acquisition site")
plt.tight_layout()

#### Think first

**Interpret the heatmap.**

1. Is missingness approximately equal across acquisition sites?
2. Can you identify assessments that appear to have been collected only at particular sites?
3. What would happen to the representation of sites if we retained only complete participants?
4. Could acquisition site be related to both missingness and diagnosis?

### Design a complete-case dataset

Suppose we want to construct an analysis table containing only participants with complete data for a selected set of core variables.

Use the missingness heatmap to choose the variables that you consider essential. The tool reports how many participants remain and whether participant retention differs across acquisition sites.

There is no universally correct set of core variables: the choice must follow the research question.

### Complete-case retention explorer

The interactive explorer lets you mark the variables you would treat as
essential and shows how many participants remain when *every* marked
variable must be recorded -- overall and for each acquisition site.

> **Interactive version on the course website.** It is embedded in the
> published Chapter 1 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_01/exercise_01.html>
> This portable notebook links to it instead of embedding it.

### Your decision

Construct two possible analysis datasets:

1. A **minimal** dataset retaining as many participants as possible.
2. A **richer** dataset containing additional behavioral information.

For each selection, record:

- the selected variables;
- the number of retained participants;
- whether retention differs between sites;
- which scientific information was sacrificed.

Which dataset would you use to investigate relationships between demographic variables and diagnosis? Would the same selection be appropriate for studying symptom severity?

<details>
<summary><strong>Check your reasoning — heatmap Q4</strong></summary>

Missingness can also depend on diagnosis. For example, some autism-specific assessments may have been administered primarily to participants in the autism group.

</details>

In [ ]:
# Can you find scores that were collected only for diagnosed participants?
missing_by_diagnosis = (
    phenotypes[top_missing_variables]
    .isna()
    .groupby(phenotypes["DX_GROUP"], observed=True)
    .mean()
    .mul(100)
)

missing_by_diagnosis.T.round(1)

### Common approaches to missing data

There is no single correct method for handling every missing value. What matters most is *why* the values are missing and what the analysis needs to do.

| Approach | When it is useful | Main caution |
| --- | --- | --- |
| **Remove** incomplete rows or a whole variable | Little data is lost, and the missingness is plausibly unrelated to the question | Shrinks the sample and can bias it if the missingness is not random |
| **Impute** the missing values — simple (mean / median / mode), model-based (e.g. KNN), or multiple imputation | A complete table is required and other variables carry enough information to estimate the gaps | Understates uncertainty and can weaken or distort relationships; parameters must be learned from training data only |
| **Keep missingness explicit** — a missing-indicator, an explicit "not recorded" category, or a method that accepts missing data | The fact that a value is missing may itself be informative | The model may learn the data-collection procedure (e.g. which site) rather than the biology |

The missingness mechanism and the goal of the analysis decide the choice; no one strategy is always best.

#### Removing incomplete observations

The simplest approach is complete-case deletion: retain only rows that have values for every required variable.

This curated table has 13 variables, and two of them — the ADOS-G and ADI-R totals — were collected for only part of the sample.

Predict before running: how many participants do you expect would remain if we required complete data for all 13 variables?

In [ ]:
core_variables = [
    "SITE_ID",
    "DX_GROUP",
    "AGE_AT_SCAN",
    "SEX",
    "FIQ",
]

retention_comparison = pd.DataFrame(
    {
        "dataset": [
            "Original dataset",
            "Complete for all 13 variables",
            "Complete for selected core variables",
        ],
        "n_participants": [
            len(phenotypes),
            len(phenotypes.dropna()),
            len(phenotypes.dropna(subset=core_variables)),
        ],
    }
)

retention_comparison["percent_retained"] = (
    retention_comparison["n_participants"]
    / len(phenotypes)
    * 100
)

retention_comparison.round(1)

#### Think first

1. Why can requiring complete data for all variables remove almost every participant?
2. Why is deletion based on a specific analysis subset more reasonable?
3. Could the retained participants differ systematically from the excluded participants?
4. What should be checked before concluding that complete-case deletion is safe?

### Imputing a numerical variable

Imputation replaces missing observations with estimated values. As a simple example, we replace missing `FIQ` values with the median observed `FIQ`, working on a copy so that the original dataframe is unchanged.

Here the median is computed from every observed value, which is fine for a one-off illustration. In a real modelling pipeline the median would be learned from the **training split only** and then applied to the validation and test data; computing it from the whole dataset first lets information from the held-out participants leak into preprocessing. The end of this section returns to this point.

In [ ]:
imputation_demo = phenotypes[
    ["SUB_ID", "SITE_ID", "DX_GROUP", "AGE_AT_SCAN", "SEX", "FIQ"]
].copy()

imputation_demo["FIQ_was_missing"] = imputation_demo["FIQ"].isna()

fiq_median = imputation_demo["FIQ"].median()
imputation_demo["FIQ_imputed"] = imputation_demo["FIQ"].fillna(fiq_median)

print(f"Median FIQ used for imputation: {fiq_median:.1f}")
print(f"Number of imputed observations: {imputation_demo['FIQ_was_missing'].sum()}")

In [ ]:
pd.DataFrame(
    {
        "Original FIQ": imputation_demo["FIQ"].describe(),
        "Median-imputed FIQ": imputation_demo["FIQ_imputed"].describe(),
    }
).round(2)

<details>
<summary><strong>Important limitation</strong></summary>

Median imputation solves the computational problem of missing values, but it does not reconstruct the unobserved measurements.

It introduces a concentration of observations at the median, usually reduces the estimated variance, and can weaken relationships between FIQ and other variables.

</details>

Imputing 99 of 1,114 `FIQ` values at the median leaves the mean almost unchanged and lowers the standard deviation only slightly, because the share of imputed rows is small. The effect grows with the fraction of missing data and with how far the imputed constant sits from the rest of the distribution — and, as the dropdown above notes, it always removes genuine variability and weakens relationships with other variables.

### Handling missing categories

For a categorical variable, possible approaches include:

- replacing missing values with the most frequent category;
- creating an explicit category such as `"Not recorded"`;
- excluding the variable or incomplete participants.

An explicit missing category is useful when the absence of a value may itself be informative. However, it may also allow a model to learn differences between sites or data-collection procedures.

In [ ]:
imputation_demo["CURRENT_MED_STATUS"] = phenotypes["CURRENT_MED_STATUS"]

imputation_demo["CURRENT_MED_STATUS_filled"] = (
    imputation_demo["CURRENT_MED_STATUS"]
    .astype("object")
    .fillna("Not recorded")
    .astype("category")
)

imputation_demo[
    ["CURRENT_MED_STATUS", "CURRENT_MED_STATUS_filled"]
].value_counts(dropna=False)

#### Think first

Suppose medication status is missing mainly at one acquisition site. What might a classification model learn from the `"Not recorded"` category?

<details>
<summary><strong>Check your reasoning</strong></summary>

The model may use `"Not recorded"` as an indirect indicator of acquisition site.

If site composition also differs between diagnostic groups, the model could appear to predict diagnosis while partly learning where the participant was scanned.

</details>

### Common approach:

For the exploratory analysis (as done here), we will not immediately impute or delete all missing values. Instead, we will:

1. preserve the original phenotype table;
2. report the available sample size for each variable;
3. examine whether missingness differs by site or diagnosis;
4. select variables according to the specific analysis;
5. perform deletion or imputation only when required.

**For later machine-learning analyses**, imputation parameters must be estimated using the training data only. Calculating an imputation value from the complete dataset before cross-validation would allow information from the test participants to influence preprocessing and would constitute data leakage.

## 5. Distributions and feature correlations

Summary statistics describe variables using only a few numbers. Visualizing their distributions can reveal information that the mean and standard deviation may hide, including:

- skewness;
- multiple peaks;
- restricted ranges;
- unusual observations;
- differences between participant groups.

We will first examine individual variables and then investigate relationships between features.

### Explore numerical distributions

A histogram divides the observed range into intervals called **bins** and counts the observations within each interval.

The choice of bin width matters. Too few bins can hide important structure, while too many bins can make random variation appear meaningful.

Use the controls below to select a variable and change the number of histogram bins.

Before changing the controls, predict:

1. What information will disappear when very few bins are used?
2. What happens when the number of bins becomes very large?
3. Will the same number of bins be suitable for age, IQ, and questionnaire scores?

The interactive histogram lets you pick a variable and change the number
of bins to see how the choice of bin width changes the shape of a
distribution without changing the data behind it. The next cell
reproduces the same kind of plot in Python.

> **Interactive version on the course website.** It is embedded in the
> published Chapter 1 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_01/exercise_01.html>
> This portable notebook links to it instead of embedding it.

The same basic plot can be produced in Python with the notebook's existing
dataframe and plotting stack.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=phenotypes,
    x="AGE_AT_SCAN",
    bins=25,
    color="steelblue",
    edgecolor="white",
    ax=ax,
)
ax.set(
    title="Distribution of age at scan",
    xlabel="Age at scan (years)",
    ylabel="Number of participants",
)
plt.show()

#### Think first

Choose at least three variables.

1. Which distribution is most symmetric?
2. Which distribution appears most skewed?
3. Which variables are discrete scores rather than continuous measurements?
4. Did changing the number of bins alter your interpretation?
5. For which variable was the KDE curve potentially misleading?

### Reading a distribution

A histogram or density plot answers questions that a mean and a standard deviation cannot:

- **Center and spread** — where do typical values sit, and how far do they range?
- **Skew** — is one tail much longer than the other? A long right tail pulls the mean above the median.
- **Multiple peaks** — could the sample be a mixture of groups measured on different scales or instruments?
- **Floor and ceiling effects** — do many observations pile up at the lowest or highest possible score?
- **Implausible values** — are there observations outside the range the measurement can take?

The interactive histogram already showed that **the number of bins changes how a distribution looks, not what was actually observed**. The same 1,114 rows are being drawn each time; only the summary changes.

The table below reports `describe()` together with the available and missing sample size for a small set of variables that behave differently from one another.

In [ ]:
# A deliberately short set: an age variable, a cognitive score, and two
# autism-assessment totals with very different completeness.
summary_variables = ["AGE_AT_SCAN", "FIQ", "SRS_TOTAL_RAW", "ADOS_G_TOTAL", "ADI_R_SOCIAL_TOTAL_A"]

distribution_summary = phenotypes[summary_variables].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
]
distribution_summary["available_n"] = phenotypes[summary_variables].notna().sum()
distribution_summary["missing_n"] = phenotypes[summary_variables].isna().sum()

distribution_summary.round(1)

#### Think first

**Read the summary.**

Use the table above.

1. For `SRS_TOTAL_RAW` the mean (about 55) is well above the median (43). Sketch the distribution shape this implies, then check it with the interactive histogram.
2. `ADOS_G_TOTAL` and `ADI_R_SOCIAL_TOTAL_A` each have fewer than 350 available observations. Does the small sample make their mean *wrong*, or does it make it *uncertain and possibly unrepresentative*? These are different problems.
3. Which of these variables are discrete (a small set of integer scores) rather than continuous?

<details>
<summary><strong>Check your reasoning</strong></summary>

1. Mean above median means a **right skew**: most participants score low, a minority score much higher, and the long upper tail drags the mean up. The interactive histogram shows a tall block near zero with a spread-out tail.
2. It makes the mean **uncertain and potentially biased**. The value is computed correctly, but it describes only the sub-sample that was measured. If the ADOS-G or ADI-R was collected mainly for particular participants, that sub-sample need not resemble the whole cohort.
3. `ADOS_G_TOTAL` and `ADI_R_SOCIAL_TOTAL_A` are sums of a fixed number of small item scores, so they take a limited set of integer values. `AGE_AT_SCAN` and the IQ scores are effectively continuous.

</details>

### Comparing a variable across groups

A common exploratory question is whether a variable *differs between groups* — here, between the two diagnostic groups. A bar chart of group means would hide both the spread and the sample size. A violin (or box) plot with the individual points drawn on top shows the whole distribution **and** how many participants sit behind it.

`DX_GROUP` is coded `1 = Autism`, `2 = Control` in the [ABIDE-II Data Legend](https://fcon_1000.projects.nitrc.org/indi/abide/ABIDEII_Data_Legend.pdf). We map the codes to readable labels on a **copy** of the columns, without altering `phenotypes`.

In [ ]:
# Readable diagnosis labels on a copy — phenotypes itself is left untouched.
diagnosis_labels = phenotypes["DX_GROUP"].astype("int64").map({1: "Autism", 2: "Control"})
group_order = ["Autism", "Control"]

fiq_by_diagnosis = pd.DataFrame(
    {"Diagnostic group": diagnosis_labels, "FIQ": phenotypes["FIQ"]}
).dropna(subset=["FIQ"])

group_counts = fiq_by_diagnosis["Diagnostic group"].value_counts()
print(
    fiq_by_diagnosis
    .groupby("Diagnostic group", observed=True)["FIQ"]
    .agg(available_n="count", median="median", mean="mean")
    .round(1)
)

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.violinplot(
    data=fiq_by_diagnosis, x="Diagnostic group", y="FIQ", order=group_order,
    hue="Diagnostic group", legend=False, palette="Set2", inner="quartile", cut=0, ax=ax,
)
# Seed only the stripplot jitter (seaborn draws it from np.random); restore
# the global RNG state afterwards so no later cell is affected.
_rng_state = np.random.get_state()
np.random.seed(0)
sns.stripplot(
    data=fiq_by_diagnosis, x="Diagnostic group", y="FIQ", order=group_order,
    color="black", size=2, alpha=0.25, jitter=0.25, ax=ax,
)
np.random.set_state(_rng_state)
ax.set_xticks(range(len(group_order)))
ax.set_xticklabels([f"{g}\n(n = {group_counts[g]})" for g in group_order])
ax.set_title("Full-scale IQ by diagnostic group (FIQ recorded for 1,015 of 1,114)")
ax.set_xlabel("")
fig.tight_layout()

The two distributions overlap heavily. The colours here are only labels — they carry no "good/bad" meaning.

A group comparison like this can be distorted by variables that differ *both* between the diagnostic groups *and* with the outcome of interest. Acquisition **site** is a prime suspect, because sites recruited different mixes of participants.

In [ ]:
# Share of each site's participants in each diagnostic group.
site_by_diagnosis = (
    pd.crosstab(phenotypes["SITE_ID"], diagnosis_labels, normalize="index")
    .mul(100)
    .loc[phenotypes["SITE_ID"].value_counts().index]  # largest sites first
)

fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(
    site_by_diagnosis[["Autism", "Control"]],
    annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=100,
    linewidths=0.5, cbar_kws={"label": "% of the site's participants"}, ax=ax,
)
ax.set_title("Diagnostic-group composition within each site")
ax.set_xlabel("Diagnostic group")
ax.set_ylabel("Acquisition site")
fig.tight_layout()

#### Think first

**Confounding and composition.**

1. The full sample is fairly balanced overall (521 Autism, 593 Control). Find two sites in the heatmap where the split is far from 50/50. What are they?
2. Two sites contributed **only** Autism participants. If age, IQ, or a scanner property differs at those sites, how could that affect a naive Autism-vs-Control comparison?
3. Name one participant-level variable, other than site, that could confound a diagnosis comparison of `FIQ`.

<details>
<summary><strong>Check your reasoning</strong></summary>

1. `ABIDEII-KKI_1` is about 27% Autism / 73% Control, while `ABIDEII-KUL_3` and `ABIDEII-NYU_2` are 100% Autism. Several smaller sites are close to 50/50.
2. Anything that varies by site — age range, IQ test used, scanner, recruitment route — becomes partly aligned with diagnosis. A difference that is really *between sites* can then show up as a difference *between diagnostic groups*. A balanced overall count does not guarantee balance within any site.
3. **Sex** is a defensible answer: in this sample females are more often in the Control group (about 70%) than males, and sex is also associated with cognitive and behavioural scores, so it can confound a group comparison. Age is another.

</details>

### Range checks and flagged values

A range check marks observations that fall outside an expected interval so a
human can look at them. Crossing the interval **flags a value for
inspection; it does not prove the value is an error.**

The usual interval is built from the quartiles. `Q1` and `Q3` are the 25th
and 75th percentiles of the variable (a quarter of the values fall below
`Q1`, a quarter above `Q3`), and the interquartile range is the distance
between them:

$$
\mathrm{IQR} = Q_3 - Q_1 .
$$

A value is flagged when it falls below the lower fence or above the upper
fence:

$$
Q_1 - 1.5\,\mathrm{IQR}
\qquad\text{and}\qquad
Q_3 + 1.5\,\mathrm{IQR}.
$$

We apply it to `AGE_AT_SCAN` and show the flagged rows using only
non-identifying fields (site, diagnosis label, age) — never `SUB_ID`.

In [ ]:
age = phenotypes["AGE_AT_SCAN"]
q1, q3 = age.quantile(0.25), age.quantile(0.75)
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr

flagged_age = phenotypes.loc[
    age.notna() & ((age < lower_fence) | (age > upper_fence)),
    ["SITE_ID", "AGE_AT_SCAN"],
].copy()
flagged_age["Diagnostic group"] = diagnosis_labels.loc[flagged_age.index]

print(f"IQR rule flags ages outside [{lower_fence:.1f}, {upper_fence:.1f}] years")
print(f"{len(flagged_age)} of {age.notna().sum()} participants flagged for inspection")
flagged_age.sort_values("AGE_AT_SCAN", ascending=False).head(8).reset_index(drop=True)

#### Think first

**Evidence before editing a value.**

The flagged ages are roughly 33 to 64 years, and many come from a small number of sites.

1. Are these values *impossible*, or just *unusual for this sample*?
2. What would you need to check before treating any of them as an error?
3. If you removed every flagged participant, what would happen to the sites they came from?

<details>
<summary><strong>Check your reasoning</strong></summary>

1. They are biologically possible ages. ABIDE-II includes adult samples; the flag reflects that most participants are children, not that these rows are wrong.
2. Cross-check the study's documented age range, confirm the coding and units (years, not months), look for a data-entry pattern, and — if in doubt — ask the people who collected the data. Any decision to drop values should be followed by a **sensitivity analysis**: repeat the key result with and without them.
3. Those sites would lose a large share of their participants, changing the sample's site and age composition. Deletion is not free.

</details>

### Pearson and Spearman correlations

Two correlation coefficients answer slightly different questions:

| | Measures | Sensitive to | Robust to |
|---|---|---|---|
| **Pearson** `r` | strength of a **linear** relationship | extreme values, non-linearity | — |
| **Spearman** `ρ` | strength of a **monotonic** relationship (via ranks) | — | outliers, non-linear but monotone shape, scale changes |

Neither establishes causation. The matrix below shows Pearson correlations for the numeric variables in the curated table. Remember that missing values differ between variables, so some cells rest on many more participants than others.

In [ ]:
# The numeric variables of the curated table.
correlation_variables = [
    "AGE_AT_SCAN", "FIQ", "VIQ", "PIQ",
    "SRS_TOTAL_RAW", "ADOS_G_TOTAL", "ADI_R_SOCIAL_TOTAL_A",
]
numeric_block = phenotypes[correlation_variables]

pearson_matrix = numeric_block.corr(method="pearson")
mask = np.triu(np.ones_like(pearson_matrix, dtype=bool), k=1)  # show lower triangle

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    pearson_matrix, mask=mask, annot=True, fmt=".2f", cmap="vlag",
    vmin=-1, vmax=1, square=True, linewidths=0.5,
    cbar_kws={"label": "Pearson r"}, ax=ax,
)
ax.set_title("Pearson correlations (lower triangle)")
fig.tight_layout()

A few things the matrix supports, read together with what we already know about the data:

- **`FIQ` correlates strongly with `VIQ` and `PIQ` (`r ≈ 0.83`).** This is not a discovery: full-scale IQ is *computed from* the verbal and performance scores, so it must track them. `VIQ` and `PIQ` are two separate sub-scores and correlate more moderately (`r ≈ 0.52`).
- **The social/communication measures move together only weakly.** `SRS_TOTAL_RAW` correlates with `ADOS_G_TOTAL` and with `ADI_R_SOCIAL_TOTAL_A` at about `r ≈ 0.30`, and only on the small group of participants who have both scores.
- **`AGE_AT_SCAN` vs `ADOS_G_TOTAL` depends on the coefficient:** Pearson `r ≈ −0.23` but Spearman `ρ ≈ −0.33`. Age is strongly right-skewed, so the rank-based coefficient tells a slightly different story; when the two disagree, look at the scatterplot.

We deliberately do **not** attach p-values or significance stars: with many pairs, some "significant" correlations would appear by chance alone and invite over-interpretation.

**Practical takeaway:** a correlation is a quick first look at how two numeric variables move together — nothing more. Before reporting one, check how many participants it rests on, whether Pearson and Spearman agree, and whether the number is definitional or driven by a difference between groups.

### Explore correlations yourself

The interactive explorer lets you choose an X and a Y variable, switch
between Pearson and Spearman, colour the points by diagnostic group or
sex, and read off the coefficient, the pairwise-complete `n`, and how
many participants were dropped for a missing value.

> **Interactive version on the course website.** It is embedded in the
> published Chapter 1 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_01/exercise_01.html>
> This portable notebook links to it instead of embedding it.

#### Think first

**Use the explorer.**

1. Set X = `FIQ`, Y = `Verbal IQ`. The correlation is very strong. Is that a fact about the brain, or about how full-scale IQ is calculated?
2. Set X = `ADOS-G total`, Y = `ADI-R social total`. What is the pairwise-complete `n`, and why is it so much smaller than for the IQ pair?
3. Set X = `FIQ`, Y = `SRS total (raw)`, method Pearson, and colour by `Diagnostic group`. Compare the overall coefficient with the two within-group coefficients. What does that tell you about the overall number?

<details>
<summary><strong>Check your reasoning</strong></summary>

1. It is mostly definitional. Full-scale IQ is a composite of verbal and performance IQ, so it must correlate strongly with each. It is not independent evidence about cognition.
2. `n` is about 152. The ADOS-G and the ADI-R are different instruments (a direct observation schedule and a parent interview); only a minority of participants were given both, so the coefficient describes a small, selected subset.
3. The overall Pearson `r` is about `−0.24`, but within the Autism group and within the Control group it is close to zero. The overall value is produced almost entirely by the two groups sitting at different places on both variables — it is a between-group difference, not a within-person association.

</details>

#### Synthesis challenge

Pick one pair of numeric variables from the curated table that you have **not** already examined, and work through the full checklist before drawing any conclusion.

1. Predict the **direction** (positive / negative / none) and the **shape** (line / cloud / curve) you expect, and write it down first.
2. Check the **missing / pairwise-complete `n`** for the pair. Is it most of the sample or a small subset?
3. Compute **Pearson and Spearman**. Do they agree? If not, look at the scatterplot and explain why.
4. Split by **diagnostic group** (or another justified grouping) and see whether the relationship survives within groups.
5. Ask whether the pair could be a **part–whole** or otherwise redundant relationship.
6. Write **two sentences**: one stating what the plot supports, and one stating what it *cannot* establish.

<details>
<summary><strong>A model reasoning process (not a single "right" answer)</strong></summary>

Suppose you chose `VIQ` and `PIQ`.

- *Prediction:* positive and roughly linear — both are IQ subscores.
- *`n`:* about 786 participants have both; a large, fairly representative subset.
- *Pearson ≈ 0.52, Spearman ≈ 0.49:* close, so the relationship is roughly linear with no single dominant outlier.
- *Within groups:* the association is similar inside the Autism and Control groups, so it is not just a between-group effect.
- *Part–whole:* `VIQ` and `PIQ` are separate subscores, so this pair is **not** a part–whole artefact — unlike either of them against `FIQ`.
- *Two sentences:* "Verbal and performance IQ are moderately, positively associated across the sample and within each diagnostic group. This is a descriptive association only: it does not tell us that one ability causes the other, and it says nothing about participants who were not tested."

</details>